# 군집화 알고리즘

https://scikit-learn.org/1.5/modules/clustering.html


![](https://scikit-learn.org/1.5/_images/sphx_glr_plot_cluster_comparison_001.png)


군집화는 머신러닝에서 데이터 포인트들을 유사한 특성을 가진 그룹으로 묶는 기법이다.

군집화는 지도학습과 달리 비지도학습(unsupervised learning)의 일종으로, 사전에 정의된 라벨 없이 데이터를 분석한다. 즉, 데이터의 내재된 구조를 파악하여 서로 유사한 데이터들을 군집(cluster)으로 묶어주는 역할을 한다.

**주요 군집화 알고리즘**
- **K-Means**: 데이터를 K개의 클러스터로 나누는 알고리즘이다. 이 알고리즘은 각 클러스터의 중심(센트로이드)까지의 거리를 최소화하는 것을 목표로 한다.
- **DBScan**: 밀도 기반의 군집화 알고리즘으로, 밀도가 높은 영역을 클러스터로 식별하며, 밀도가 낮은 데이터 포인트는 노이즈로 간주한다. 이를 통해 클러스터의 모양에 제약 없이 다양한 형태의 군집을 탐지할 수 있다.
- **Mean-Shift**: 데이터의 밀도가 높은 지역으로 이동하면서 클러스터를 형성하는 알고리즘이다. 이 알고리즘은 클러스터의 개수를 미리 지정할 필요가 없다는 점이 특징이다.
- **Gaussian Mixture Model (GMM)**: 데이터를 여러 개의 가우시안 분포의 조합으로 모델링하여 클러스터링을 수행하는 알고리즘이다. 각 클러스터에 속할 확률을 계산하여 데이터를 분류한다.

**군집화의 적용 분야**
- **고객, 마켓, 브랜드, 사회 경제 활동 세분화(Segmentation)**: 군집화 기법을 활용해 고객의 특성에 따라 그룹을 나누고, 마켓 및 브랜드 전략을 세우는 데 활용한다.

- **이미지 검출 및 트래킹(Image Segmentation and Tracking)**: 이미지 데이터에서 특정 객체를 검출하고 분할하거나 추적하는 데 사용된다.

- **이상 탐지(Anomaly Detection)**: 정상적인 데이터와 다른 이상 데이터를 탐지하는 데 활용한다.


# k-평균
K-Means는 비지도 학습의 군집화 기법으로, 데이터 포인트를 K개의 군집으로 묶는 데 사용된다.

각 군집은 유사한 데이터 포인트를 하나로 묶기 위해 **중심점(centroid)** 을 계산한다.

![](https://upload.wikimedia.org/wikipedia/commons/e/ea/K-means_convergence.gif?20170530143526)

**작동단계**
1. K개의 중심점을 임의로 선택한다.
2. 각 데이터 포인트를 가장 가까운 중심점에 할당하여 군집을 형성한다.
3. 각 군집의 데이터 포인트를 기반으로 새로운 중심점을 계산한다.
4. 2~3단계를 반복하여, __중심점의 변화가 거의 없을 때까지 실행한다.__
5. 최종적으로 모든 데이터가 가장 가까운 중심점에 할당되며, 모든 중심점은 최적의 위치에 놓이게 된다.

**특징**
- 장점: 간단한 개념과 구현, 빠른 계산 속도
- 단점: K 값을 미리 정해야 함, 군집이 원형 구조가 아닐 경우 성능 저하 가능성

**장점**
- **일반적인 군집화에서 가장 많이 활용되는 알고리즘**
- **알고리즘이 쉽고 간결**
- **대용량 데이터에도 활용** 가능

**단점**
- **거리 기반 알고리즘으로 속성의 개수가 매우 많을 경우 군집화 정확도가 떨어질 수 있다**(이를 위해 PCA로 차원 축소를 적용해야 할 수도 있다).
- **반복을 수행하는데, 반복 횟수가 많을 경우 수행 시간이 느려진다**.
- **이상치(Outlier) 데이터에 취약**하다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.datasets import load_iris

# 붓꽃
X, y = load_iris(return_X_y=True, as_frame=True)

## 최적의 k 찾기
- elbow method
- silouette score
  
### Elbow Method
클러스터 개수가 늘어날수록 각 클러스터 내부의 거리합(Inertia, SSE(오차 제곱합))은 작아진다.
하지만 어느 순간부터 급격한 감소가 완만해지는 지점이 있다.
그 지점을 **팔꿈치(Elbow)** 라고 부르며, 이 지점의 k가 적절하다.

절차:
1. k를 1부터 N까지 변화시키며 KMeans 모델을 학습.
2. 각 k에 대한 SSE (Within-cluster Sum of Squares)를 계산.
3. k vs SSE 그래프를 그려서 꺾이는 지점을 찾는다.

In [ ]:
# 붓꽃
X, y = load_iris(return_X_y=True, as_frame=True)

### Silhouette Score
클러스터 간의 분리도와 클러스터 내부의 응집도를 동시에 평가한 값.
값이 클수록 클러스터링 품질이 좋다는 뜻. 일반적으로 값이 가장 큰 k가 최적이다.

- 실루엣 점수 범위: -1 ~ 1
- 1에 가까울수록 군집이 잘 형성된 것
- 0에 가까우면 군집이 모호함
- 음수는 잘못된 군집화

$$ s(i) = \frac{b(i)−a(i)}{max(a(i),b(i))} $$

- a(i)는 데이터 포인트 i가 속한 클러스터 내의 평균 거리(클러스터 응집력).
- b(i)는 데이터 포인트 i가 다른 가장 가까운 클러스터와의 평균 거리(클러스터 분리도).
이 때, 전체 데이터에 대한 실루엣 점수는 모든 데이터 포인트의 실루엣 점수의 평균으로 계산된다.


**장점**
- 군집의 품질을 직관적으로 평가할 수 있으며, 개별 데이터 포인트 수준에서의 평가가 가능하다.
- 다양한 군집화 알고리즘에 적용 가능하다.
  
**단점**
  - 실루엣 점수가 높은 것이 항상 좋은 군집을 의미하는 것은 아니다. 군집의 수와 데이터의 구조에 따라 해석에 주의가 필요하다.
